# Post processing and visualizing mandyoc data

This notebook reads the NetCDF files created by tapioca (`meshes2netcdf.jl`): `temperature.nc`, `density.nc`, `strain.nc`, `surface.nc`, ... in the scenario folder.

The following notebook allows the user to visualize a given mandyoc output (e.g.: `'temperature'`, `'strain_rate'`, `'lithology'`, etc) according to a given number of time instants (e.g.: `instants = [0, 10, 20, 30]` Myr).

For a given instant in the `instants` the script will find the closest time step to this instant and plot the respective output.

In [ ]:
#increase the display size to 95% of window width
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
display(HTML("<style>.output_result { max-width:95% !important; }</style>"))


import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import interp1d
import glob
import os
import pylab as pl
import astropy.units as u
import astropy.table as tb
from astropy.table import QTable
import mpl_toolkits.axisartist
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import string

path = os.getcwd().split('/')
machine_path = '/'+path[1]+'/'+path[2] #cat the /home/user/ or /Users/user from system using path

import sys
sys.path.append(f"{machine_path}/opt/post_processing") #functions/mandyocIO.py, if the notebook is not run from post_processing
from functions.mandyocIO import plot_lithology_legend, add_colorbar #legend and colorbars with fixed size [inches]

##File of fancy plot style. To use des.mplstyle file you must have an LaTeX distribution in your system
#plt.style.use(machine_path+'/Mestrado/cenarios/mandyoc/scripts_gera_inputs_imgs/new_outputs/des.mplstyle')

In [ ]:
_datasets = {} #datasets already opened: {path: xarray.Dataset}

def open_nc(fpath, prop):
    '''
    Open (only once) the NetCDF file created by tapioca for a Mandyoc property.

    Parameters
    ----------

    fpath: str
        Path to the scenario directory (with the .nc files).

    prop: str
        Property of the mandyoc outputs: temperature, density, strain, strain_rate, viscosity, heat, pressure or surface.
    '''

    fname = fpath + prop + ".nc"
    if(fname not in _datasets):
        _datasets[fname] = xr.open_dataset(fname)

    return _datasets[fname]

def step_index(dataset, step):
    '''
    Return the time index of a given step in a dataset created by tapioca.
    '''

    return int(np.where(dataset.step.values == int(step))[0][0])

def read_params(fpath):
    '''
    Read the mesh of the .nc files created by tapioca and returns Nx, Nz, Lx, Lz of the scenario givem by fpath
    
    Parameters
    ----------
    
    fpath: str
        Path to the scenario directory (with the .nc files).
    '''
    
    dataset = open_nc(fpath, "temperature") #tapioca: x from 0 to Lx and z from -Lz to 0 [m]
    Nx = int(dataset.sizes["x"])
    Nz = int(dataset.sizes["z"])
    Lx = float(dataset.x.values[-1] - dataset.x.values[0])
    Lz = float(dataset.z.values[-1] - dataset.z.values[0])

    return Nx, Nz, Lx, Lz


def read_data(prop, step, fpath):
    '''
    Read data from the .nc file created by tapioca and return a (Nx, Nz) array.
    
    Parameters
    ----------
    
    prop: str
        Property of the mandyoc outputs: temperature, density, strain, strain_rate, viscosity, heat or pressure.
        
    step: str
        Time step of numerical scenario.
        
    fpath: str
        Path to the scenario directory (with the .nc files).
    '''
    
    dataset = open_nc(fpath, prop)
    TT = dataset[prop].isel(time=step_index(dataset, step)).values.astype(float) #(Nz, Nx)
    TT[np.abs(TT)<1.0E-200]=0
    TTT = np.transpose(TT) #(Nx, Nz)

    return TTT
        
def read_density(fpath, step, Nx, Nz):
    '''
    Read density from density.nc (created by tapioca) and return a (Nz, Nx) array.
    
    Parameters
    ----------
    fpath: str
        Path to the scenario directory (with the .nc files).
        
    step: str
        Time step of numerical scenario.
        
    Nx: int
        Number of points in x direction.
        
    Nz: int
        Number of points in z direction.
    '''

    dataset = open_nc(fpath, "density")
    Rho = dataset.density.isel(time=step_index(dataset, step)).values.astype(float)

    return Rho

def read_temperature(fpath, step, Nx, Nz):
    '''
    Read temperature from temperature.nc (created by tapioca) and return a (Nz, Nx) array
    
    Parameters
    ----------
    fpath: str
        Path to the scenario directory (with the .nc files).
        
    step: str
        Time step of numerical scenario.
        
    Nx: int
        Number of points in x direction.
        
    Nz: int
        Number of points in z direction.
    '''

    dataset = open_nc(fpath, "temperature")
    Temper = dataset.temperature.isel(time=step_index(dataset, step)).values.astype(float)

    return Temper

def read_surface(fpath, step):
    '''
    Read the surface from surface.nc (created by tapioca) and return it [km] in the x of the mesh.
    surface.nc has more points than the mesh, so it is interpolated.
    Returns None if there is no surface.nc (surface not tracked in the scenario).

    Parameters
    ----------
    fpath: str
        Path to the scenario directory (with the .nc files).

    step: str
        Time step of numerical scenario.
    '''

    if(not os.path.isfile(fpath + "surface.nc")):
        return None

    Nx, Nz, Lx, Lz = read_params(fpath)
    dataset = open_nc(fpath, "surface")
    topo = dataset.surface.isel(time=step_index(dataset, step)).values/1.0E3
    topo = np.interp(np.linspace(0, Lx/1000.0, Nx), dataset.x.values/1.0E3, topo)

    return topo

def extract_interface(z, Z, Nx, field_datai, value_to_search):
    '''
    Extract interface from Rhoi according to a given density (rho)

    Parameters
    ----------
    z: array_like
        Array representing z direction.

    Z: array_like
        Array representing z direction resampled with higher resolution.

    Nx: int
        Number of points in x direction.

    field_data: array_like (Nz, Nx)
        Mandyoc field from mandyoc (e.g.: density, temperature, etc).

    value_to_search: float
        Value of density to be searched in field_data array.

    Returns
    -------
    mapped_interface: array_like
        Interface extracted from Rhoi field
    '''

    mapped_interface = []

    for j in np.arange(Nx):

        interp_function = interp1d(z, field_datai[:,j]) #return a "function" of interpolation to apply in other array
        
        interface_inverted = interp_function(Z)[::-1] #inverted array: from top to bottom

        idx = np.argmax(interface_inverted > value_to_search) #first occurrence of rho in the inverted array
        idx_corected = len(interface_inverted) - idx #correcting the index to the original array

        depth = Z[idx_corected]

        mapped_interface = np.append(mapped_interface, depth)

    return mapped_interface

def read_lithology(fpath, step):
    '''
    Read lithology.nc (created by tapioca) and return the Mandyoc layer codes (order of the layers in interfaces.txt,
    0 = bottom layer) in a (Nzl, Nxl) array, first row at the bottom. The lithology mesh is 5 times finer than the
    mesh of the other fields: Nxl = 5*(Nx-1)+1 and Nzl = 5*(Nz-1)+1.

    Parameters
    ----------
    fpath: str
        Path to the scenario directory (with the .nc files).

    step: str
        Time step of numerical scenario.
    '''

    dataset = open_nc(fpath, "lithology")
    litho = dataset.lithology.isel(time=step_index(dataset, step)).values

    return litho

def measure_crust_from_lithology(fpath, step, crust_codes):
    '''
    Measure the crust in each column of the mesh counting the cells of the crust in lithology.nc, as done for the
    oceanic crust (thickness_oceanic_crust) in frames_generator_litho_melt_thickness.py: each cell of the lithology
    mesh has dz/5 of thickness.

    Parameters
    ----------
    fpath: str
        Path to the scenario directory (with the .nc files).

    step: str
        Time step of numerical scenario.

    crust_codes: list of int
        Mandyoc layer codes of the crust.

    Returns
    -------
    topography_interface: array_like
        Top of the crust [km] corrected from air layer (NaN where there is no crust).

    lower_crust_interface: array_like
        Base of the crust [km] corrected from air layer (NaN where there is no crust).

    total_crustal_thickness: array_like
        Crustal thickness [km] (sum of the crust cells in the column).
    '''

    Nx, Nz, Lx, Lz = read_params(fpath)
    h_air = 40.0 #km
    dz = Lz/(Nz-1)

    litho = read_lithology(fpath, step)[:, ::5] #one column of each 5: the columns in the x of the mesh
    cond = np.isin(litho, crust_codes)

    total_crustal_thickness = np.sum(cond, axis=0)/(5/dz)/1000 #km

    z_litho = np.linspace(-Lz/1000.0, 0, litho.shape[0])
    has_crust = np.any(cond, axis=0)
    idx_top = litho.shape[0] - 1 - np.argmax(cond[::-1, :], axis=0) #last crust cell from the bottom
    idx_base = np.argmax(cond, axis=0) #first crust cell from the bottom

    topography_interface = np.where(has_crust, z_litho[idx_top] + h_air, np.nan)
    lower_crust_interface = np.where(has_crust, z_litho[idx_base] + h_air, np.nan)

    return topography_interface, lower_crust_interface, total_crustal_thickness

def measure_crustal_thickness(fpath, step, Nx, Nz, Lx, Lz, x_begin=600.0, x_end=900.0, crust_codes=[4, 5]):
    '''
    Measure the crustal thickness in a given region of the model, counting the cells of the crust in lithology.nc
    (see measure_crust_from_lithology).

    Parameters
    ----------
    fpath: str
        Path to the scenario directory.

    step: int
        Step of Mandyoc output files.
        
    Nx: int
        Number of points in x direction.

    Nz: int
        Number of points in z direction.

    Lx: float
        Length of the model in x direction.

    Lz: float
        Length of the model in z direction.

    x_begin: float
        Initial x position to measure the crustal thickness.

    x_end: float
        Final x position to measure the crustal thickness.

    crust_codes: list of int
        Mandyoc layer codes of the crust.

    Returns
    -------

    position_max_thickness: float
        Position of the maximum crustal thickness in the selected region.

    topo_max_thickness: float
        Topography value in the maximum crustal thickness in the selected region.

    crustal_thickness: float
        Maximum crustal thickness in the selected region [km]

    lower_crust_interface: array_like
        Lower crust interface [km] corrected from air layer

    topography_interface: array_like
        Topography interface [km] corrected from air layer

    total_crustal_thickness: array_like
        Crustal thickness [km]

    '''
    x_aux = np.linspace(0, Lx/1000.0, Nx)
    
    topography_interface, lower_crust_interface, total_crustal_thickness = measure_crust_from_lithology(fpath, step, crust_codes)
    
    #selecting the region to measure the crustal thickness
    cond_x = (x_aux >= x_begin) & (x_aux <= x_end)

    x_region = x_aux[cond_x]  
    topo_region = topography_interface[cond_x]
    crustal_thickness_region = total_crustal_thickness[cond_x]

    #index of the maximum crustal thickness
    idx = np.where(crustal_thickness_region == np.max(crustal_thickness_region))[0][0]

    #information of the maximum crustal thickness
    position_max_thickness = x_region[idx]
    topo_max_thickness = topo_region[idx]
    crustal_thickness = crustal_thickness_region[idx]
    

    return position_max_thickness, topo_max_thickness, crustal_thickness, lower_crust_interface, topography_interface, total_crustal_thickness

def measure_margins_width(fpath, step, Nx, Nz, Lx, Lz, xl_begin=600, xl_end=800, xr_begin=800, xr_end=1000, crust_codes=[4, 5]):
    '''
    Measure the width of the rifted margins in a given region of the model.

    Parameters
    ----------

    fpath: str
        Path to the scenario directory.

    step: int
        Step of Mandyoc output files.

    Nx: int
        Number of points in x direction.

    Nz: int
        Number of points in z direction.

    Lx: float
        Length of the model in x direction.

    Lz: float
        Length of the model in z direction.

    xl_begin: float
        Initial x position to measure the left margin.

    xl_end: float
        Final x position to measure the left margin.

    xr_begin: float
        Initial x position to measure the right margin.

    xr_end: float
        Final x position to measure the right margin.

    Returns
    -------
    marginl_begin: float
        Initial position of the left margin.

    marginl_end: float
        Final position of the left margin.

    marginl_wdt: float
        Width of the left margin.

    marginr_begin: float
        Initial position of the right margin.

    marginr_end: float
        Final position of the right margin.

    marginr_wdt: float
        Width of the right margin.
    '''
    x = np.linspace(0, Lx/1000.0, Nx)

    topography_interface, lower_interface, crustal_thickness = measure_crust_from_lithology(fpath, step, crust_codes)
    
    #left side
    cond_xl = (x >= xl_begin) & (x <= xl_end)

    # topol_region = topography_interface[cond_xl]
    # idx_max_topol = np.where(topol_region == np.max(topol_region))[0][0] #the first index of the maximum topography is more distal
    # marginl_begin = x[cond_xl][idx_max_topol]

    marginl_begin = xl_begin

    # left_margin_region = lower_interface[cond_xl]
    left_margin_region = crustal_thickness[cond_xl]

    # idx_min_lithol = np.where(lithol_region == np.max(lithol_region))[0][0]
    # idx_min_lithol = np.where(left_margin_region >= -5.0)[0][0]
    idx_min_lithol = np.where(left_margin_region <=5.0)[0][0]
    marginl_end = x[cond_xl][idx_min_lithol]

    marginl_wdt = np.abs(marginl_end - marginl_begin)

    #right side
    cond_xr = (x >= xr_begin) & (x <= xr_end)

    # topor_region = topography_interface[cond_xr]
    # idx_max_topor = np.where(topor_region == np.max(topor_region))[0][-1] #the last index of the maximum topography is more distal
    # marginr_begin = x[cond_xr][idx_max_topor]
    marginr_begin = xr_end

    # right_margin_region = lower_interface[cond_xr]
    right_margin_region = crustal_thickness[cond_xr]
    # idx_min_lithor = np.where(lithor_region == np.max(lithor_region))[0][-1]
    # idx_min_lithor = np.where(right_margin_region >= -5.0)[0][-1]
    idx_min_lithor = np.where(right_margin_region <=5.0)[0][-1]
    marginr_end = x[cond_xr][idx_min_lithor]

    marginr_wdt = np.abs(marginr_end - marginr_begin)

    return marginl_begin, marginl_end, marginl_wdt, marginr_begin, marginr_end, marginr_wdt

def find_nearest(array, value):
    '''Return the index in array nearest to a given value.
    
    Parameters
    ----------
    
    array: array_like
        1D array used to find the index
        
    value: float
        Value to be seached
    '''
    
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

def read_times_steps(fpath):
    '''Read the times and steps of the .nc files created by tapioca
       
       Parameters
       ----------
       fapth: str
           Path to scenario directory.
    '''
    
    dataset = open_nc(fpath, "temperature")
    times = np.round(dataset.time.values, 2) #Myr
    steps = dataset.step.values.astype('int')
    return times, steps

def trim_axs(axs, N):
    '''Little helper to index the axs as 1D list
       
       Parameters
       ----------
       axs: `~matplotlib.axes.Axes` list of Axes
           
       N: int
           Total number of subplots.
    '''
    
    axs = axs.flat
    for ax in axs[N:]:
        ax.remove()
    return axs[:N]


def plot_data(fpath, ax, axs, prop, step, xlims, ylims):
    '''
    Plot data from mandyoc according to a given property.

    Parameters
    ----------
    fapth: str
           Path to scenario directory.
    
    ax: `~matplotlib.axes._subplots.AxesSubplot` Axis subplot.
    
    axs: 
        Numpy array of `~matplotlib.axes.Axes` list of Axes after use of trim_axis function.

    prop: str
        Property from mandyoc.     

    step: int
        Step of Mandyoc output files .

    xlims: list
        List with the limits of x axis
        
    ylims: list
        List with the limits of y axis
    '''
    
    #label of colorbars
#     print(prop)
    props_label = {'density':              r'$\mathrm{[kg/m^3]}$',
                   'heat':                 'log(W/kg)',
                   'lithology':            r'log$(\epsilon_{II})$',
                   'pressure':             'P [GPa]',
                   'strain':               r'Accumulated strain [$\varepsilon$]',
                   'strain_rate':          r'log($\dot{\varepsilon}$)',
                   'temperature':          r'$^{\circ}\mathrm{[C]}$',
                   'temperature_anomaly':  r'Temperature anomaly $^{\circ}\mathrm{[C]}$',
                   'topography':           'Topography [km]',
                   'viscosity':            'log(Pa.s)',
                   'vs_anomaly':           r'dVs/Vs [\%]'
                   }
    
    props_cmap = {'density': 'viridis',
                  'heat': 'inferno',
                  'lithology': 'viridis',
                  'pressure': 'viridis',
                  'strain': 'viridis', #Default. Comment this line and uncomment one of the options bellow
#                   'strain': 'cividis',
#                   'strain': 'Greys',
#                   'strain': 'inferno',
#                   'strain': 'magma',
                  'strain_rate': 'viridis',
                  'temperature': 'viridis',
                  'temperature_anomaly': 'RdBu_r',
                  'topography': '',
                  'viscosity': 'viridis'
                   }

    #limits of colorbars
    vals_minmax = {'density':             [0.0, 3378.],
                   'heat':                [np.log10(1.0E-13), np.log10(1.0E-9)],
                   'lithology':           [None, None],
                   'pressure':            [-1.0E-3, 1.0],
                   'strain':              [-3.0, 1.8],
                   'strain_rate':         [np.log10(1.0E-19), np.log10(1.0E-12)],
                   'temperature':         [0, 1600],
                   'temperature_anomaly': [-150, 150],
                   'topography':          [-6, 6],
                   # 'viscosity':           [np.log10(1.0E16), np.log10(1.0E25)],
                   # 'viscosity':           [np.log10(1.0E22), np.log10(1.0E25)],
                   'viscosity':           [np.log10(1.0E18), np.log10(1.0E25)]
                  }

    Nx, Nz, Lx, Lz = read_params(fpath)

    ##Creating a grid to plot
    xi = np.linspace(0, Lx/1000, Nx)
    zi = np.linspace(-Lz/1000+40, 0+40, Nz) #km, +40 to compensate the air layer above sea level
    xx, zz = np.meshgrid(xi, zi)

#             idx = np.where(tab['Steps']==stp[0])[0][0]

    #Removing the mean uplift between of left side
    #condx = (xi >= 100) & (xi <= 600)
    # z_mean = np.mean(topography[condx])
    z_mean = 40.0 #np.mean(topo[condx])
    
    topo = read_surface(fpath, step)
    if(topo is not None):
        topo = topo + 40
        condx = (xi >= 100) & (xi <= 400)
        mean = np.mean(topo[condx])
        topoi = topo - np.abs(mean)
    

    val_minmax = vals_minmax[prop]
    
    # if(plot_thickness == True):
    #     x_pos, topo_max_thickness, crustal_thickness = measure_crustal_thickness(fpath, step, Nx, Nz, Lx, Lz, x_begin, x_end, topoi)
    #     ax.plot([x_pos, x_pos], [topo_max_thickness, -crustal_thickness], color='black', linestyle='--', linewidth=1.0, zorder=52)
    #     ax.text(x_pos+100, -crustal_thickness/2, f'{str(round(crustal_thickness, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)
        
    if(prop == 'lithology'):
        data = read_data('strain', step, fpath)
        
    elif(prop == 'temperature_anomaly'):
        data = read_data('temperature', step, fpath)
        
    elif(prop == 'topography'):
        topo_from_density = True
        
        if(topo_from_density == True):
            Rhoi = read_density(fpath, step, Nx, Nz)
            interfaces=[2900, 3365]
            ##Extract layer topography
            z = np.linspace(Lz/1000.0, 0, Nz)
            Z = np.linspace(Lz/1000.0, 0, 8001) #zi
            x = np.linspace(Lx/1000.0, 0, Nx)

            topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
            topo_interface -= np.abs(z_mean)
            topo_interface = -1.0*topo_interface

            data = topo_interface
            
        else:
            data = topoi
        
    else:
        data = read_data(prop, step, fpath)
    data[data==0] = 1.0E-28 #avoid log(0)
    
    if(prop == 'lithology' or prop == "strain" or prop == "strain_rate" or prop == "viscosity" or prop == "heat"):
        data = np.log10(data)
    
    #PLOT DATA
    
    if(prop != 'lithology' and prop != 'strain' and prop != 'pressure' and prop != 'temperature_anomaly' and prop != 'topography'): #properties that need a colorbar
        im = ax.imshow(np.transpose(data),
                       cmap = props_cmap[prop], origin='lower',
                       extent = (0, Lx / 1.0E3, -Lz / 1.0E3 + 40, 0 + 40),
                       vmin = val_minmax[0],
                       vmax = val_minmax[1],
                       aspect = 'auto')

        Rhoi = read_density(fpath, step, Nx, Nz)
        interfaces=[2900, 3365]
        ##Extract layer topography
        z = np.linspace(Lz/1000.0, 0, Nz)
        Z = np.linspace(Lz/1000.0, 0, 8001) #zi
        x = np.linspace(Lx/1000.0, 0, Nx)
            
        topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
        topo_interface -= np.abs(z_mean)
        topo_interface = -1.0*topo_interface
        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        cs = ax.contour(xx, zz, Temperi, 100, levels=isotherms, linewidths = 1.0, colors=['red', 'red', 'red'])
        
        if(ax==axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
        
#         ax.contour(xx, zz, Rhoi, 100, levels=interfaces, colors=['black', 'red'])

        if(ax == axs[-1]):
        
            ticks = np.linspace(val_minmax[0], val_minmax[1], 5, endpoint=True)
            clb = add_colorbar(fig, ax, im, props_label[prop], position=(0.02, 0.25), length=2.4, ticks=ticks, fmt='%.0f') #fixed size [inches]
            
        xaux = xx[0]
        condaux = (xaux>xlims[0]) & (xaux<xlims[1])
        xaux = xaux[condaux]
#         ax.fill_between(xaux, topoi[condaux], 38.5, color='white', alpha=1.0, zorder=51)
        ax.fill_between(xaux, topo_interface[condaux], 35.5, color='white', alpha=1.0, zorder=51)
        
    elif(prop == 'strain'):
        im = ax.imshow(np.transpose(data),
                       cmap = props_cmap[prop], origin='lower',
                       extent = (0, Lx / 1.0E3, -Lz / 1.0E3 + 40, 0 + 40),
                       vmin = val_minmax[0],
                       vmax = val_minmax[1],
                       aspect = 'auto')

        Rhoi = read_density(fpath, step, Nx, Nz)
        interfaces=[2900, 3365]
        ##Extract layer topography
        z = np.linspace(Lz/1000.0, 0, Nz)
        Z = np.linspace(Lz/1000.0, 0, 8001) #zi
        x = np.linspace(Lx/1000.0, 0, Nx)
            
        topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
        topo_interface -= np.abs(z_mean)
        topo_interface = -1.0*topo_interface
        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        cs = ax.contour(xx, zz, Temperi, 100, levels=isotherms, linewidths=1.0, colors=['red', 'red', 'red'])
        
        if(ax==axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
        
#         ax.contour(xx, zz, Rhoi, 100, levels=interfaces, colors=['black', 'red'])
        
        ticks = np.linspace(np.min(data), np.max(data), 5, endpoint=True)
        clb = add_colorbar(fig, ax, im, props_label[prop], position=(0.02, 0.25), length=2.4, ticks=ticks, fmt='%.1f') #fixed size [inches]
            
        xaux = xx[0]
        condaux = (xaux>xlims[0]) & (xaux<xlims[1])
        xaux = xaux[condaux]
#         ax.fill_between(xaux, topoi[condaux], 38.5, color='white', alpha=1.0, zorder=51)
        ax.fill_between(xaux, topo_interface[condaux], 35.5, color='white', alpha=1.0, zorder=51)
    
    elif(prop == 'pressure'): #GPa 
        im = ax.imshow(np.transpose(data/1.0E9),
                       cmap=props_cmap[prop],
                       origin='lower',
                       extent=(0, Lx / 1.0E3, -Lz / 1.0E3 + 40, 0 + 40),
                       vmin = val_minmax[0],
                       vmax = val_minmax[1],
                       aspect = 'auto')
        
        if(ax == axs[-1]):
            ticks = np.linspace(val_minmax[0], val_minmax[1], 5, endpoint=True)
            clb = add_colorbar(fig, ax, im, props_label[prop], position=(0.02, 0.25), length=2.4, ticks=ticks, fmt='%.3f') #fixed size [inches]
        
        Rhoi = read_density(fpath, step, Nx, Nz)
        interfaces=[2900, 3365]
        ##Extract layer topography
        z = np.linspace(Lz/1000.0, 0, Nz)
        Z = np.linspace(Lz/1000.0, 0, 8001) #zi
        x = np.linspace(Lx/1000.0, 0, Nx)
        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        cs = ax.contour(xx, zz, Temperi, 100, levels=isotherms, linewidths=1.0, colors=['red', 'red', 'red'])
        
        if(ax==axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
        
        topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
        topo_interface -= np.abs(z_mean)
        topo_interface = -1.0*topo_interface
        xaux = xx[0]
        condaux = (xaux>xlims[0]) & (xaux<xlims[1])
        xaux = xaux[condaux]
        
        ax.fill_between(xaux, topo_interface[condaux], 35.5, color='white', alpha=1.0, zorder=51)
        
    elif(prop == 'temperature_anomaly'):
        #removing horizontal mean temperature
        A = data  #shape: (Nx, Nz)
        B = np.transpose(A) #shape: (Nz, Nx)
        C = np.mean(B, axis=1) #shape: 151==Nz
        D = (B.T - C) #B.T (Nx,Nz) para conseguir subtrair C
        
        im = ax.imshow(np.transpose(D), 
                       cmap = props_cmap[prop], 
                       origin = 'lower',
                       extent = (0, Lx / 1.0E3, -Lz / 1.0E3 + 40, 0 + 40), 
                       vmin = val_minmax[0], 
                       vmax = val_minmax[1],
                       aspect = 'auto'
                       )

        if(ax == axs[-1]):
            ticks = np.linspace(val_minmax[0], val_minmax[1], 5, endpoint=True)
            clb = add_colorbar(fig, ax, im, props_label[prop], position=(0.02, 0.25), length=2.4, ticks=ticks, fmt='%.0f') #fixed size [inches]
        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        
        cs = ax.contour(xx,
                        zz,
                        Temperi,
                        100,
                        levels=isotherms,
                        colors=['red', 'red', 'red'])
        
        if(ax==axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
    elif(prop == 'topography'):
        ax.plot(xx[0], data, alpha = 1, linewidth = 2.0, color = "blueviolet")
        
    elif(prop == 'lithology'): #shaded lithology plot
        cr = 255.
        color_sed = (241./cr,184./cr,68./cr)
        color_dec = (137./cr,81./cr,151./cr)
        color_uc = (228. / cr, 156. / cr, 124. / cr)
        color_lc = (240. / cr, 209. / cr, 188. / cr)
        color_lit = (155. / cr, 194. / cr, 155. / cr)
        color_mlit_uc = 'xkcd:grass green' #(228. / cr, 156. / cr, 124. / cr)
        color_mlit_lc = 'xkcd:darkish green' #(240. / cr, 209. / cr, 188. / cr)
        color_ast = (207. / cr, 226. / cr, 205. / cr)
        
        Rhoi = read_density(fpath, step, Nx, Nz)
        interfaces=[2900, 3365]
        
        ##Extract layer topography
        z = np.linspace(Lz/1000.0, 0, Nz)
        Z = np.linspace(Lz/1000.0, 0, 8001) #zi
        x = np.linspace(Lx/1000.0, 0, Nx)
            
        topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
        topo_interface -= np.abs(z_mean)
        topo_interface = -1.0*topo_interface
        
        ax.contourf(xx,
                    zz,
                    Rhoi,
                    #You must choose intermediate values according to the reference densities
                    #adopted in interfaces.txt file, otherwise the colors will not be displayed
                    #correctly
                    levels = [200.,
                              2350, #sed
                              2450, #dec
                              2750, #uc
                              2900, #lc
                              3315, #mlit upper craton
                              3350, #mlit lower craton
                              3365, #mlit non cratonic
                              3900, #asthenosphere
                              ],
                    colors = [
                              color_sed,
                              color_dec,
                              color_uc,
                              color_lc,
                              color_mlit_uc,
                              color_mlit_lc,
                              color_lit,
                              color_ast],
                    # levels = [200., 2350, 2450, 2700, 2800, 3354, 3900],
                    # colors = [color_dec, color_sed, color_uc, color_lc, color_lit, color_ast]
                    )
        
        im=ax.imshow(np.transpose(data),
                     cmap = 'Greys',
                     origin = 'lower',
                     extent = (0, Lx / 1.0E3, -Lz / 1.0E3 + 40,0 + 40),
                     zorder = 50,
                     alpha = 0.2, vmin=-0.5,
                     vmax = 0.7,
                     aspect = 'auto')


        if(ax == axs[-1]):
            #lithology/strain legend with fixed size [inches], whatever the size of the figure.
            #Bands from top to bottom: densities 2700, 2800, 3300, 3350, 3400 in the levels of the contourf above
            plot_lithology_legend(ax, colors=[color_uc, color_lc, color_mlit_uc, color_lit, color_ast],
                                  vmin=-0.5, vmax=0.7, alpha=0.2, position=(0.98, 0.09), loc='lower right')

        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        cs = ax.contour(xx, zz, Temperi, 100, levels=isotherms, linewidths=0.5, colors=['red', 'red', 'red'])
        
        if(ax==axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
        
        xaux = xx[0]
        condaux = (xaux>xlims[0]) & (xaux<xlims[1])
        xaux = xaux[condaux]

#         ax.fill_between(xaux, topoi[condaux], 38.5, color='white', alpha=1.0, zorder=51)
        ax.fill_between(xaux, topo_interface[condaux], 35.5, color='white', alpha=1.0, zorder=51)
    
    else:
        im = ax.imshow(np.transpose(data),
                       cmap = props_cmap[prop],
                       origin = 'lower',
                       extent = (0, Lx / 1.0E3, -Lz / 1.0E3 + 40, 0 + 40),
                       vmin = val_minmax[0],
                       vmax = val_minmax[1],
                       aspect = 'auto')
    
        Rhoi = read_density(fpath, step, Nx, Nz)
        interfaces = [2900, 3365]
        
        ##Extract layer topography
        z = np.linspace(Lz/1000.0, 0, Nz)
        Z = np.linspace(Lz/1000.0, 0, 8001) #zi
        x = np.linspace(0, Lx/1000.0, Nx)
        
        Temperi = read_temperature(fpath, step, Nx, Nz)
        isotherms=[550, 800, 1300]
        cs = ax.contour(xx, zz, Temperi, 100, levels=isotherms, colors=['red', 'red', 'red'])
        
        if(ax == axs[0]):
            fmt = {}
            for level, isot in zip(cs.levels, isotherms):
                fmt[level] = str(level) + r'$^{\circ}$C'

            ax.clabel(cs, cs.levels, fmt=fmt, inline=True, use_clabeltext=True)
            
        topo_interface = extract_interface(z, Z, Nx, Rhoi, 200.) #200 kg/m3 = air/crust interface
        topo_interface -= np.abs(z_mean)
        topo_interface = -1.0 * topo_interface
        
        xaux = xx[0]
        condaux = (xaux > xlims[0]) & (xaux < xlims[1])
        xaux = xaux[condaux]
        ax.fill_between(xaux, topo_interface[condaux], 35.5, color='white', alpha=1.0, zorder=51)

# How to plot

## In the following cell, you must inform:
 - The path to your `scenario` (folder with the `.nc` files created by tapioca) without `/OS_home/username` because this is collected by `machine_path`.
     - Ex.: `/mandyoc/experiments/my_experiment/`
 - The mandyoc property `prop` that you want to visualize uncommenting one the commented options.
     - `prop = 'lithology'` is the default.
 - The values in list `instants` containing the instants that you want to visualize.
     - Ex: `instants = [0, 10, 20, 30]` #Myr

In [ ]:
machine_path

In [ ]:
#Set scenario path

#scenario = '/Documents/aguiasala/M45/M45/'
#scenario = '/Documents/Simulations/M60_1x1/'
#scenario = '/Documents/Simulations/Invertedmargin/'
# scenario = '/Documents/Simulations/MR_dk/'
# scenario = '/Desktop/teste/'
# scenario = '/Desktop/teste3/'
scenario = '/Desktop/teste4/'
# scenario = '/opt/teste/'
##############################################################################################################
#Creating path to the experiment

fpath = machine_path + scenario
print(fpath)

##############################################################################################################
#Set mandyoc property

# prop = 'density'
# prop = 'heat'
prop = 'lithology'
#prop = 'pressure'
#prop = 'strain'
# prop = 'strain_rate'
#prop = 'temperature'
# prop = 'temperature_anomaly'
# prop = 'topography'
# prop = 'viscosity'

##############################################################################################################
#Mandyoc layer codes (order of the layers in interfaces.txt, 0 = bottom layer) of the crust,
#used to measure the crustal thickness counting the crust cells of lithology.nc

crust_codes = [4, 5, 6, 7, 8, 9] #double keel: lower crust 1, seed, lower crust 2, upper crust, decolement, sediments
# crust_codes = [4, 5] #accordion (AR) and rifting: lower crust, upper crust
# crust_codes = [4, 5, 6] #accordion (AR) with the basaltic (oceanic) crust

##############################################################################################################
#Set instants

#instants = [25,75,105]
#instants = [0,25, 35, 49,120]
# instants = [0,5,10,15]
# instants = [30, 100]
instants = [25, 75, 175, 230]
# instants = [0, 30, 60, 90, 120] #Myr
# instants = [0, 30, 60, 90, 120, 150]

##############################################################################################################
#Reading scenario basic informations

times, steps = read_times_steps(fpath)
Nx, Nz, Lx, Lz = read_params(fpath)
x = np.linspace(0, Lx/1000, Nx)
z = np.linspace(-Lz/1000.0, 0, Nz)

##############################################################################################################
#Set domain limits of the figure
#Aqui mudamos os limites de x e y de plotagem da figura!
if(prop != 'topography'):
    xlims = [0, Lx / 1.0e3]
    ylims = [-Lz / 1.0e3 + 40, 40]
    # ylims = [-80, 40]
else:
    xmin = 0 #+ 200
    xmax = Lx / 1.0E3 #- 200
    xlims = [xmin, xmax]
    ylims = [-6, 6]

##############################################################################################################
#Creating figure and subplots
plt.close('all')

if(len(instants) == 1):
    fig, axs = plt.subplots(len(instants), 1, constrained_layout=True, figsize=(12,6), sharex=True, sharey=True)
    axs = [axs]
else:
    fig, axs = plt.subplots(len(instants), 1, constrained_layout=True, figsize=(12,12), sharex=True, sharey=True)
    axs = trim_axs(axs, len(instants))


alphabet = list(string.ascii_lowercase)
letters = alphabet[0: len(instants)]

##############################################################################################################
#Plot data
plot_thickness = True
# plot_thickness = False

for instant, ax in zip(instants, axs):
    idx = find_nearest(times, instant)
    step = steps[idx]
    # plot_data(fpath, ax, axs, prop, step, xlims, ylims)
    plot_data(fpath, ax, axs, prop, step, xlims, [ylims])

    if(plot_thickness == True):
        if(instant == 25): #instant to measure margin width
            x_begin = 200.0
            x_end = 700.0
            calc_margin_width = True
            
        elif(instant == 230): #instant to measure margin width
            x_begin = 200.0
            x_end = 700.0
            # x_begin = 900.0
            # x_end = 1400.0
            calc_margin_width = True

        else: #orogeny instants
            x_begin = 200.0
            x_end = 800.0
            calc_margin_width = False
            
        x_pos, topo_max_thickness, local_crustal_thickness, lc_base, topography, total_crustal_thickness = measure_crustal_thickness(fpath, step, Nx, Nz, Lx, Lz, x_begin, x_end, crust_codes=crust_codes)
        
        # ax.plot(np.linspace(0, Lx/1000, Nx), lc_base, color='black', linestyle='--', linewidth=2.0, zorder=52)
        # ax.plot(np.linspace(0, Lx/1000, Nx), topo, color='black', linestyle='-', linewidth=2.0, zorder=52)
        ax.plot([x_pos, x_pos], [topo_max_thickness, -local_crustal_thickness], color='black', linestyle='--', linewidth=1.0, zorder=52)
        ax.text(x_pos+100, -local_crustal_thickness/2, f'{str(round(local_crustal_thickness, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

        if(instant == 230 or instant == 25): #checking for regions with crustal thickness between 7 and 10 km
            h_crust = total_crustal_thickness #crustal thickness
            cond_thickness = (h_crust <= 10) & (h_crust >= 7)
            ax.plot(x[cond_thickness], lc_base[cond_thickness], 'o', color='xkcd:orange', zorder=52)

        if(calc_margin_width == True): #Measuring margin width
            if(instant == 25):
                xl_begin = 780
                xl_end = 900

                xr_begin = 950
                xr_end = 1100
                
            elif(instant == 230):
                xl_begin = 300
                xl_end = 750

                xr_begin = 800
                xr_end = 1350

            marginl_begin, marginl_end, marginl_len, marginr_begin, marginr_end, marginr_len = measure_margins_width(fpath, step, Nx, Nz, Lx, Lz, xl_begin, xl_end, xr_begin, xr_end, crust_codes=crust_codes)

            ax.plot([marginl_begin, marginl_end], [10,10], color='orange', linestyle='-', linewidth=4.0, zorder=52)
            ax.text(marginl_begin + marginl_len/2, 12, f'{str(round(marginl_len, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

            ax.plot([marginr_begin, marginr_end], [10, 10], color='blue', linestyle='-', linewidth=4.0, zorder=52)
            ax.text(marginr_begin - marginr_len/2, 12, f'{str(round(marginr_len, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

    #print(steps[idx], times[idx], instant)
    
##############################################################################################################


#Setting plot details

fsize = 14
axs[-1].set_xlabel('x [km]', fontsize=fsize)
for ax, letter, instant in zip(axs, letters, instants):
    label_text = letter + ') ' + str(instant) + ' Myr'
    ax.text(0.015,
            0.95,
            label_text,
            color='k',
            bbox=dict(edgecolor = 'white',
                      facecolor = 'white',
                      alpha = 0.6),
            fontsize=fsize,
            horizontalalignment='left',
            verticalalignment='center',
            transform=ax.transAxes,
            zorder=52)
    
    if(prop != 'topography'):
        ax.set_ylabel('Depth [km]', fontsize=fsize)
        ax.tick_params(axis='both', labelsize=fsize)
        axs[-1].tick_params(axis='both', labelsize=fsize)
        ax.set_yticks(np.linspace(ylims[0], 0, 5))
    else:
        ax.set_ylabel('Topogarphy [km]', fontsize=fsize)
        ax.tick_params(axis='both', labelsize=fsize)
        axs[-1].tick_params(axis='both', labelsize=fsize)
        ax.set_yticks(np.linspace(ylims[0], ylims[1], 7))
        ax.grid('-k', alpha = 0.7)

axs[0].set_xlim(xlims)
axs[0].set_ylim(ylims)
# axs[0].set_ylim([-40,40])    
##############################################################################################################
#Saving figure

figname = scenario.split('/')[-2] + '_'+prop
print(figname)

#This will save inside the notebook directory
#fig.savefig(figname+'.pdf', dpi=400)
#fig.savefig(figname+'.png', dpi=400)

#This will save inside the experiment directory
#fig.savefig(fpath+figname+'.pdf', dpi=400)
fig.savefig(fpath+figname+'.png', dpi=400)
#fig.savefig(fpath+figname+'.jpeg', dpi=400)

In [ ]:
idx = find_nearest(times, 230)
step = steps[idx]

Nx, Nz, Lx, Lz = read_params(fpath)

x = np.linspace(0, Lx/1000.0, Nx)

#crust measured counting the crust cells of lithology.nc (crust_codes set above)
topography_interface, lc_interface, h_crust = measure_crust_from_lithology(fpath, step, crust_codes)

#base of the lithospheric mantle from the density field
z = np.linspace(-Lz/1000.0, 0, Nz)
Z = np.linspace(-Lz/1000.0, 0, 8001) #zi
Rhoi = read_density(fpath, step, Nx, Nz)
h_air = 40.0 # km
lower_interface = extract_interface(z, Z, Nx, Rhoi, 3365.) + h_air

plt.figure(figsize=(12,6))

cond_thickness = (h_crust <= 10) & (h_crust >= 5)
plt.plot(x[cond_thickness], lc_interface[cond_thickness], '.', color='xkcd:orange', zorder=52, label='5 < Hc < 10 km')

plt.plot(x, topography_interface, color='black', linestyle='-', linewidth=2.0, zorder=52, label='Top of the crust')
plt.plot(x, lc_interface, color='blue', linestyle='-', linewidth=2.0, zorder=52, label='Base of the crust')
plt.plot(x, lower_interface, color='green', linestyle='-', linewidth=2.0, zorder=52, label='Mantle Lithosphere')
plt.legend(loc='lower center', fontsize=12)
plt.xlabel('x [km]')
plt.ylabel('Depth [km]')
plt.ylim([-150, 40])
plt.xlim([0, 1600])


print(h_crust[cond_thickness])


In [ ]:
Rhoi

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([5, 5, 5, 5, 5])
np.abs(a-b)